# Devoir 2 — Système RAG pour l'exploitation du Code de la Route Marocain

**Objectif** : Concevoir et implémenter un système RAG (Retrieval-Augmented Generation) permettant de répondre à des questions en langage naturel à partir du Code de la Route marocain.


**Date** : Mai 2026

---
**Pipeline du système :**
1. Préparation & nettoyage des données (`code_route.csv`)
2. Chunking intelligent des articles
3. Génération d'embeddings + indexation FAISS
4. Retriever sémantique
5. Intégration LLM (Qwen2.5) + prompt engineering
6. Comparaison de 3 LLMs
7. Évaluation (précision, recall, F1, ROUGE)
8. Détection de questions hors domaine (out-of-domain)
9. Interface utilisateur Gradio

## 1. Installation des dépendances

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate gradio rouge-score scikit-learn pandas numpy

## 2. Chargement et préparation des données

On charge le fichier `code_route.csv` produit dans le Devoir 1.

Colonnes clés utilisées : `article_id`, `article_header`, `texte_brut`, `amende_fixe`, `amende_min`, `amende_max`, `points_retrait`, `categorie_vehicule`.

In [ ]:
import pandas as pd
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# Chargement du CSV
df = pd.read_csv("code_route.csv")
print(f"📁 Données chargées : {len(df)} lignes, {df.shape[1]} colonnes")
df.head(3)

In [ ]:
# Aperçu des colonnes
print("\n📊 Colonnes disponibles :")
for col in df.columns:
    print(f"  - {col}")
    
# Vérification des valeurs manquantes
print("\n⚠️ Valeurs manquantes :")
print(df.isnull().sum())

### 2.1 Nettoyage et normalisation du texte

Le nettoyage inclut :
- Suppression des diacritiques arabes (tashkeel)
- Normalisation Unicode
- Suppression des espaces superflus

In [ ]:
def clean_arabic_text(text: str) -> str:
    """Normalise le texte arabe : supprime les diacritiques, espaces superflus."""
    if not isinstance(text, str):
        return ""
    # Normalisation Unicode
    text = unicodedata.normalize("NFC", text)
    # Suppression des diacritiques arabes (harakat)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)
    # Suppression des caractères spéciaux indésirables
    text = re.sub(r'[\u200c\u200d]', ' ', text)
    # Normalisation des espaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Application du nettoyage
df["texte_clean"] = df["texte_brut"].apply(clean_arabic_text)
df["article_header_clean"] = df["article_header"].apply(clean_arabic_text)

# Suppression des doublons
initial_count = len(df)
df = df.drop_duplicates(subset=["texte_clean"]).reset_index(drop=True)
print(f"🗑️ Doublons supprimés : {initial_count - len(df)}")

# Suppression des lignes avec texte trop court
df = df[df["texte_clean"].str.len() > 20].reset_index(drop=True)

print(f"✅ Après nettoyage : {len(df)} articles uniques")

# Statistiques sur les longueurs
df["longueur_clean"] = df["texte_clean"].str.len()
print("\n📏 Statistiques des longueurs d'articles :")
print(df["longueur_clean"].describe().to_string())

### 2.2 Chunking — Découpage en unités exploitables

Les articles longs sont découpés en chunks chevauchants pour éviter de perdre du contexte aux frontières.

In [ ]:
def chunk_text(text: str, max_chars: int = 500, overlap: int = 100) -> list:
    """
    Découpe un texte long en chunks avec chevauchement.
    
    Args:
        text: Le texte à découper
        max_chars: Taille maximale de chaque chunk
        overlap: Nombre de caractères de chevauchement
    """
    if len(text) <= max_chars:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        end = min(start + max_chars, len(text))
        
        # Essayer de couper à la fin d'une phrase si possible
        if end < len(text):
            # Chercher un point, point d'exclamation ou point d'interrogation
            for sep in ['۔', '!', '؟', '.', ';']:
                last_period = text.rfind(sep, start, end)
                if last_period != -1 and last_period > start:
                    end = last_period + 1
                    break
        
        chunks.append(text[start:end].strip())
        
        if end >= len(text):
            break
            
        start = end - overlap
    
    return chunks

# Création des chunks
chunks_data = []

for _, row in df.iterrows():
    chunks = chunk_text(row["texte_clean"])
    for i, chunk in enumerate(chunks):
        chunks_data.append({
            "article_id": row["article_id"],
            "article_header": row["article_header_clean"],
            "chunk_id": i,
            "chunk_text": chunk,
            "amende_fixe": row.get("amende_fixe", 0) if pd.notna(row.get("amende_fixe", 0)) else 0,
            "amende_min": row.get("amende_min", 0) if pd.notna(row.get("amende_min", 0)) else 0,
            "amende_max": row.get("amende_max", 0) if pd.notna(row.get("amende_max", 0)) else 0,
            "points_retrait": row.get("points_retrait", 0) if pd.notna(row.get("points_retrait", 0)) else 0,
            "categorie_vehicule": row.get("categorie_vehicule", "Tous") if pd.notna(row.get("categorie_vehicule", "Tous")) else "Tous",
        })

chunks_df = pd.DataFrame(chunks_data)

# Texte enrichi pour l'embedding (header + chunk)
chunks_df["embed_text"] = chunks_df["article_header"] + " | " + chunks_df["chunk_text"]

print(f"📄 Total chunks générés : {len(chunks_df)}")
print(f"📊 En moyenne : {len(chunks_df)/len(df):.1f} chunks par article")

# Aperçu
chunks_df.head(3)

## 3. Embeddings et indexation FAISS

On utilise un modèle multilingue capable de traiter l'arabe : `paraphrase-multilingual-MiniLM-L12-v2`

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import time

# Modèle multilingue
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"🔄 Chargement du modèle d'embeddings : {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = chunks_df["embed_text"].tolist()

print("📈 Génération des embeddings...")
start_time = time.time()
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings, dtype="float32")
elapsed = time.time() - start_time

print(f"✅ Embeddings générés en {elapsed:.2f} secondes")
print(f"📐 Shape des embeddings : {embeddings.shape}")

In [ ]:
# Construction de l'index FAISS avec similarité cosinus
# Normalisation L2 pour utiliser Inner Product comme similarité cosinus
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = cosinus après normalisation
index.add(embeddings)

print(f"🎯 Index FAISS créé avec succès")
print(f"   - Dimensions: {dimension}")
print(f"   - Vecteurs indexés: {index.ntotal}")
print(f"   - Type: IndexFlatIP (similarité cosinus)")

# Sauvegarde de l'index pour usage futur
faiss.write_index(index, "code_route_index.faiss")
chunks_df.to_csv("chunks_data.csv", index=False)
print("💾 Index et données sauvegardés")

## 4. Retriever sémantique

La fonction `retrieve()` encode la question, cherche les k chunks les plus proches dans FAISS, et retourne les passages avec leurs métadonnées.

In [ ]:
def retrieve(query: str, k: int = 3) -> list:
    """
    Retrieve the top-k most relevant chunks for a query.
    
    Args:
        query: Question de l'utilisateur
        k: Nombre de chunks à récupérer
        
    Returns:
        Liste des résultats avec scores et métadonnées
    """
    # Encodage de la requête
    query_vec = embed_model.encode([query], show_progress_bar=False)
    query_vec = np.array(query_vec, dtype="float32")
    faiss.normalize_L2(query_vec)
    
    # Recherche dans FAISS
    scores, indices = index.search(query_vec, min(k, index.ntotal))
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0 and idx < len(chunks_df):
            row = chunks_df.iloc[idx]
            results.append({
                "article_id": int(row["article_id"]) if pd.notna(row["article_id"]) else None,
                "article_header": row["article_header"],
                "chunk_text": row["chunk_text"],
                "score": float(score),
                "amende_fixe": float(row["amende_fixe"]) if pd.notna(row["amende_fixe"]) else 0,
                "amende_min": float(row["amende_min"]) if pd.notna(row["amende_min"]) else 0,
                "amende_max": float(row["amende_max"]) if pd.notna(row["amende_max"]) else 0,
                "points_retrait": int(row["points_retrait"]) if pd.notna(row["points_retrait"]) else 0,
                "categorie_vehicule": row["categorie_vehicule"],
            })
    
    return results

# Test rapide
print("🔍 Test du retriever :")
test_results = retrieve("رخصة السياقة", k=3)
for r in test_results:
    print(f"   [Score={r['score']:.3f}] Article {r['article_id']}: {r['article_header']}")
    print(f"      Extrait: {r['chunk_text'][:100]}...")
    print()

## 5. Intégration LLM + Prompt Engineering

Le prompt injecte les documents récupérés comme contexte et demande au modèle de répondre uniquement à partir de ce contexte.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

def load_llm(model_name: str):
    """
    Load a causal LLM and return a generation pipeline.
    
    Args:
        model_name: Nom du modèle sur HuggingFace
        
    Returns:
        Pipeline de génération de texte
    """
    print(f"🔄 Chargement du modèle : {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    # Configuration du tokenizer si nécessaire
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
    
    gen_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=300,
        do_sample=False,
        temperature=0.7,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    return gen_pipeline

# Chargement du LLM principal (léger pour l'exécution rapide)
generator_qwen = load_llm("Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
def build_prompt(query: str, docs: list) -> str:
    """
    Construit un prompt RAG structuré avec le contexte récupéré.
    
    Args:
        query: Question de l'utilisateur
        docs: Documents récupérés
        
    Returns:
        Prompt formaté pour le LLM
    """
    context_parts = []
    
    for i, doc in enumerate(docs, 1):
        meta = f"[Article {doc['article_id']}]"
        if doc.get("amende_fixe", 0) > 0:
            meta += f" | Amende fixe: {doc['amende_fixe']} MAD"
        if doc.get("amende_max", 0) > 0:
            meta += f" | Amende: {doc['amende_min']}-{doc['amende_max']} MAD"
        if doc.get("points_retrait", 0) > 0:
            meta += f" | Points retirés: {doc['points_retrait']}"
            
        context_parts.append(
            f"--- Source {i} ---\n{meta}\n{doc['article_header']}\n{doc['chunk_text']}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""<|im_start|>system
Vous êtes un assistant expert spécialisé dans le Code de la Route marocain.
Vous devez répondre UNIQUEMENT en utilisant les informations contenues dans le contexte fourni.
Si la réponse ne se trouve pas dans le contexte, dites clairement que vous ne pouvez pas répondre.
Citez toujours la source (numéro d'article) de vos informations.
Répondez en français ou en arabe selon la langue de la question.
<|im_end|>
<|im_start|>user
Contexte :
{context}

Question : {query}

Veuillez répondre en vous basant UNIQUEMENT sur le contexte ci-dessus.
<|im_end|>
<|im_start|>assistant
"""
    return prompt


def rag_answer(query: str, generator, k: int = 3) -> dict:
    """
    Pipeline RAG complet : retrieve → build prompt → generate.
    
    Args:
        query: Question de l'utilisateur
        generator: Pipeline de génération LLM
        k: Nombre de documents à récupérer
        
    Returns:
        Dictionnaire avec réponse, sources et métadonnées
    """
    # Récupération des documents
    docs = retrieve(query, k=k)
    
    if not docs:
        return {
            "query": query,
            "answer": "Désolé, aucun document pertinent n'a été trouvé pour répondre à cette question.",
            "sources": [],
            "top_scores": [],
        }
    
    # Construction du prompt
    prompt = build_prompt(query, docs)
    
    # Génération
    try:
        output = generator(prompt, max_new_tokens=300)
        full_text = output[0]["generated_text"]
        # Extraction de la réponse après le marqueur assistant
        if "<|im_start|>assistant" in full_text:
            answer = full_text.split("<|im_start|>assistant")[-1].strip()
        elif "Assistant:" in full_text:
            answer = full_text.split("Assistant:")[-1].strip()
        else:
            answer = full_text[len(prompt):].strip() if len(full_text) > len(prompt) else full_text
    except Exception as e:
        print(f"Erreur de génération: {e}")
        answer = "Une erreur s'est produite lors de la génération de la réponse."
    
    return {
        "query": query,
        "answer": answer,
        "sources": [f"Art. {d['article_id']}: {d['article_header']}" for d in docs],
        "top_scores": [d["score"] for d in docs],
        "raw_docs": docs,
    }

# Test du pipeline
print("🧪 Test du pipeline RAG :")
result = rag_answer("ما هي شروط الحصول على رخصة السياقة؟", generator_qwen)
print(f"Question: {result['query']}")
print(f"Sources: {result['sources']}")
print(f"Scores: {result['top_scores']}")
print(f"\nRéponse:\n{result['answer']}")

## 6. Détection de questions hors domaine (Out-of-Domain)

Si le score de similarité maximal est en dessous d'un seuil, la question est considérée hors domaine.

In [ ]:
OOD_THRESHOLD = 0.25  # Seuil de similarité cosinus

def is_out_of_domain(query: str, threshold: float = OOD_THRESHOLD) -> tuple:
    """
    Vérifie si une question est hors domaine.
    
    Args:
        query: Question à vérifier
        threshold: Seuil de similarité
        
    Returns:
        (bool, max_score): True si hors domaine, et le score maximum
    """
    docs = retrieve(query, k=1)
    if not docs:
        return True, 0.0
    max_score = docs[0]["score"]
    return max_score < threshold, max_score


def rag_with_ood(query: str, generator, k: int = 3) -> dict:
    """
    Pipeline RAG avec détection hors domaine.
    
    Args:
        query: Question de l'utilisateur
        generator: Pipeline LLM
        k: Nombre de documents à récupérer
        
    Returns:
        Dictionnaire avec réponse et métadonnées OOD
    """
    is_ood, max_score = is_out_of_domain(query)
    
    if is_ood:
        return {
            "query": query,
            "answer": "⚠️ Cette question ne semble pas être liée au Code de la Route marocain. "
                      "Je suis spécialisé uniquement dans les questions relatives au code de la route. "
                      f"(Similarité maximale: {max_score:.3f} < seuil {OOD_THRESHOLD})",
            "sources": [],
            "top_scores": [],
            "ood": True,
            "max_score": max_score,
        }
    
    result = rag_answer(query, generator, k)
    result["ood"] = False
    result["max_score"] = max_score
    return result

# Tests
print("🔍 Test de détection hors domaine :")
print("-" * 50)

test_queries = [
    "ما هو الحد الأقصى للسرعة في المدينة؟",  # Dans le domaine
    "Quelle est la capitale de la France?",   # Hors domaine
    "كم تبلغ غرامة تجاوز السرعة؟",            # Dans le domaine
    "Who won the World Cup in 2022?",         # Hors domaine
]

for q in test_queries:
    r = rag_with_ood(q, generator_qwen)
    print(f"\nQ: {q}")
    print(f"Score max: {r['max_score']:.3f}")
    print(f"OOD: {r['ood']}")
    print(f"Réponse: {r['answer'][:150]}..." if len(r['answer']) > 150 else f"Réponse: {r['answer']}")

## 7. Comparaison de 3 LLMs

On compare les réponses de trois modèles sur un ensemble de questions de référence.

In [ ]:
# Chargement des deux modèles supplémentaires
print("\n" + "="*60)
print("CHARGEMENT DES MODÈLES POUR COMPARAISON")
print("="*60)

try:
    generator_qwen_15 = load_llm("Qwen/Qwen2.5-1.5B-Instruct")
except Exception as e:
    print(f"⚠️ Impossible de charger Qwen2.5-1.5B-Instruct: {e}")
    generator_qwen_15 = None

try:
    # Alternative plus légère si le 1.5B n'est pas disponible
    generator_qwen_base = load_llm("Qwen/Qwen2.5-0.5B")
except Exception as e:
    print(f"⚠️ Impossible de charger Qwen2.5-0.5B: {e}")
    generator_qwen_base = None

In [ ]:
import time

# Registre des LLM disponibles
LLM_REGISTRY = {}
LLM_REGISTRY["Qwen2.5-0.5B-Instruct"] = generator_qwen
if generator_qwen_15:
    LLM_REGISTRY["Qwen2.5-1.5B-Instruct"] = generator_qwen_15
if generator_qwen_base:
    LLM_REGISTRY["Qwen2.5-0.5B-Base"] = generator_qwen_base

print(f"📋 Modèles chargés: {list(LLM_REGISTRY.keys())}")

# Questions d'évaluation
EVAL_QUESTIONS = [
    "ما هي الوثائق الضرورية لاستخراج رخصة السياقة؟",
    "ما هي عقوبة قيادة السيارة بدون رخصة؟",
    "ما هو الحد الأقصى للسرعة على الطريق السيار؟",
    "كم تبلغ غرامة استعمال الهاتف أثناء السياقة؟",
]

comparison_results = []

print("\n" + "="*60)
print("COMPARAISON DES MODÈLES")
print("="*60)

for idx, question in enumerate(EVAL_QUESTIONS, 1):
    print(f"\n📝 Question {idx}: {question}")
    print("-" * 40)
    
    row = {"question": question}
    
    for llm_name, gen in LLM_REGISTRY.items():
        print(f"\n🔄 Test de {llm_name}...")
        t0 = time.time()
        
        try:
            res = rag_answer(question, gen, k=3)
            elapsed = time.time() - t0
            row[f"{llm_name}_answer"] = res["answer"]
            row[f"{llm_name}_time"] = round(elapsed, 2)
            row[f"{llm_name}_sources"] = res["sources"]
            print(f"   ✓ Terminé en {elapsed:.2f}s")
        except Exception as e:
            print(f"   ✗ Erreur: {e}")
            row[f"{llm_name}_answer"] = f"[ERREUR: {str(e)[:100]}]"
            row[f"{llm_name}_time"] = -1
            row[f"{llm_name}_sources"] = []
    
    comparison_results.append(row)

comparison_df = pd.DataFrame(comparison_results)

# Affichage des résultats
print("\n" + "="*60)
print("RÉSULTATS COMPLETS")
print("="*60)

for _, r in comparison_df.iterrows():
    print(f"\n{'='*60}")
    print(f"❓ Question: {r['question']}")
    for llm_name in LLM_REGISTRY:
        time_col = f"{llm_name}_time"
        answer_col = f"{llm_name}_answer"
        if time_col in r and answer_col in r:
            print(f"\n🤖 [{llm_name}] ({r[time_col]}s)")
            print(f"   {r[answer_col][:400]}")
            if len(r[answer_col]) > 400:
                print("   ...")

# Sauvegarde des résultats
comparison_df.to_csv("llm_comparison_results.csv", index=False)
print("\n💾 Résultats sauvegardés dans 'llm_comparison_results.csv'")

## 8. Évaluation des performances (Précision, Recall, F1, ROUGE)

On utilise un jeu de questions-réponses de référence (gold standard) pour évaluer la qualité du système.

In [ ]:
from rouge_score import rouge_scorer
import json

# Gold standard: (question, articles_relevants, reponse_reference)
GOLD_STANDARD = [
    {
        "question": "هل يمكن لأجنبي السياقة في المغرب برخصته الأجنبية؟",
        "relevant_article_ids": [2, 4],
        "reference_answer": "يجوز للأجانب الحاملين لرخصة سياقة أجنبية سارية المفعول السياقة في المغرب خلال مدة صلاحية الرخصة. كما يمكنهم الحصول على رخصة سياقة مغربية بناء على رخصتهم الأصلية."
    },
    {
        "question": "ما هي شروط الحصول على رخصة سياقة دولية؟",
        "relevant_article_ids": [4],
        "reference_answer": "تسلم الهيئات المؤهلة رخصة دولية للسياقة وفق الاتفاقية الدولية للسير على الطرق. يجب أن تكون رخصة السياقة الوطنية سارية المفعول."
    },
    {
        "question": "ما هي الفترة المسموح بها للمغاربة المقيمين بالخارج للسياقة برخصتهم الأجنبية؟",
        "relevant_article_ids": [2],
        "reference_answer": "سنة واحدة ابتداء من تاريخ إقامتهم بالمغرب."
    },
    {
        "question": "ما هي عقوبة قيادة السيارة بدون رخصة؟",
        "relevant_article_ids": [3, 5],
        "reference_answer": "القيادة بدون رخصة تعتبر مخالفة خطيرة يعاقب عليها بغرامة مالية تتراوح بين 1000 و 5000 درهم و حجز المركبة لمدة قد تصل إلى 30 يوما."
    },
]

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

eval_rows = []

print("="*60)
print("ÉVALUATION DES PERFORMANCES")
print("="*60)

for i, sample in enumerate(GOLD_STANDARD, 1):
    print(f"\n📋 Échantillon {i}: {sample['question'][:50]}...")
    
    # Récupération des documents
    docs = retrieve(sample["question"], k=5)
    retrieved_ids = [d["article_id"] for d in docs]
    relevant_ids = sample["relevant_article_ids"]
    
    # Métriques de retrieval
    if retrieved_ids:
        tp = len(set(retrieved_ids) & set(relevant_ids))
        precision = tp / len(retrieved_ids)
        recall = tp / len(relevant_ids) if relevant_ids else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    else:
        precision = recall = f1 = 0
    
    print(f"   📊 Retrieval - Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")
    print(f"   📄 Articles récupérés: {retrieved_ids}")
    print(f"   ✅ Articles pertinents: {relevant_ids}")
    
    # Qualité de génération
    try:
        gen_result = rag_answer(sample["question"], generator_qwen, k=3)
        rouge = scorer.score(sample["reference_answer"], gen_result["answer"])
        rouge_l = rouge["rougeL"].fmeasure
        print(f"   📝 ROUGE-L: {rouge_l:.3f}")
    except Exception as e:
        print(f"   ⚠️ Erreur calcul ROUGE: {e}")
        rouge_l = 0
    
    eval_rows.append({
        "question": sample["question"][:40] + "..." if len(sample["question"]) > 40 else sample["question"],
        "retrieval_precision": round(precision, 3),
        "retrieval_recall": round(recall, 3),
        "retrieval_f1": round(f1, 3),
        "rouge_L": round(rouge_l, 3),
    })

eval_df = pd.DataFrame(eval_rows)

print("\n" + "="*60)
print("RÉSULTATS DÉTAILLÉS")
print("="*60)
print(eval_df.to_string(index=False))

print("\n" + "="*60)
print("MOYENNES GLOBALES")
print("="*60)
means = eval_df[["retrieval_precision", "retrieval_recall", "retrieval_f1", "rouge_L"]].mean().round(3)
for metric, value in means.items():
    print(f"📊 {metric}: {value}")

# Sauvegarde
eval_df.to_csv("evaluation_results.csv", index=False)
print("\n💾 Résultats sauvegardés dans 'evaluation_results.csv'")

## 9. Interface utilisateur — Gradio

Interface web simple permettant d'interroger le système RAG avec sélection du LLM et affichage des sources.

In [ ]:
import gradio as gr

def gradio_rag(question: str, model_choice: str, top_k: int) -> tuple:
    """
    Interface Gradio pour le RAG.
    
    Args:
        question: Question de l'utilisateur
        model_choice: Modèle LLM sélectionné
        top_k: Nombre de documents à récupérer
        
    Returns:
        (réponse, sources_formatées)
    """
    if not question or not question.strip():
        return "Veuillez entrer une question.", ""
    
    # Sélection du générateur
    if model_choice in LLM_REGISTRY and LLM_REGISTRY[model_choice] is not None:
        gen = LLM_REGISTRY[model_choice]
    else:
        gen = generator_qwen
    
    # Appel du pipeline RAG avec détection OOD
    result = rag_with_ood(question, gen, k=top_k)
    
    answer = result["answer"]
    
    # Formatage des sources
    if result["ood"] or not result["sources"]:
        sources_text = "❌ Aucune source pertinente trouvée."
    else:
        sources_parts = []
        for i, src in enumerate(result["sources"], 1):
            # Récupérer le document complet pour plus de détails
            doc = result.get("raw_docs", [{}])[i-1] if result.get("raw_docs") and len(result["raw_docs"]) >= i else {}
            score = result["top_scores"][i-1] if result["top_scores"] and len(result["top_scores"]) >= i else 0
            
            sources_parts.append(f"""
### 📌 {src}
- **Score de similarité**: {score:.3f}
- **Extrait**: {doc.get('chunk_text', '')[:300]}...
""")
        sources_text = "\n".join(sources_parts)
    
    return answer, sources_text


# Configuration de l'interface
with gr.Blocks(title="RAG Code de la Route Marocain", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🚗 Assistant du Code de la Route Marocain
    
    Bienvenue sur l'assistant intelligent basé sur le **Retrieval-Augmented Generation (RAG)**.
    
    Ce système permet de poser des questions en langage naturel (arabe ou français) sur le Code de la Route marocain.
    Les réponses sont générées à partir des articles de loi extraits dans le Devoir 1.
    
    ---
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="💬 Votre question",
                placeholder="Exemples:\n- ما هي شروط الحصول على رخصة السياقة؟\n- Quelle est l'amende pour excès de vitesse ?\n- هل يمكن السياقة برخصة أجنبية في المغرب؟",
                lines=3
            )
        
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=list(LLM_REGISTRY.keys()),
                value=list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct",
                label="🤖 Modèle LLM"
            )
            topk_slider = gr.Slider(
                minimum=1, maximum=7, step=1, value=3,
                label="📚 Nombre de documents récupérés (Top-K)"
            )
    
    submit_btn = gr.Button("🔍 Interroger", variant="primary", size="lg")
    
    with gr.Row():
        with gr.Column():
            answer_output = gr.Textbox(
                label="✨ Réponse générée",
                lines=8,
                interactive=False
            )
        
        with gr.Column():
            sources_output = gr.Markdown(label="📖 Sources et références")
    
    submit_btn.click(
        fn=gradio_rag,
        inputs=[question_input, model_selector, topk_slider],
        outputs=[answer_output, sources_output]
    )
    
    # Exemples
    gr.Markdown("---\n### 📝 Exemples de questions")
    
    gr.Examples(
        examples=[
            ["ما هي شروط الحصول على رخصة السياقة؟", list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct", 3],
            ["هل يجوز لأجنبي السياقة برخصته في المغرب؟", list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct", 3],
            ["كم تبلغ غرامة استعمال الهاتف أثناء القيادة؟", list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct", 3],
            ["Quelle est l'amende pour excès de vitesse sur autoroute ?", list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct", 3],
            ["What is the capital of France?", list(LLM_REGISTRY.keys())[0] if LLM_REGISTRY else "Qwen2.5-0.5B-Instruct", 3],
        ],
        inputs=[question_input, model_selector, topk_slider],
        label="Cliquez sur un exemple pour le tester"
    )

# Lancement de l'interface
print("\n" + "="*60)
print("🚀 LANCEMENT DE L'INTERFACE GRADIO")
print("="*60)
demo.launch(share=True)

## 10. Résumé et Conclusions

### ✅ Fonctionnalités implémentées

1. **Préparation des données**
   - Nettoyage et normalisation du texte arabe
   - Suppression des diacritiques
   - Découpage intelligent en chunks avec chevauchement

2. **Indexation et recherche**
   - Embeddings multilingues avec `paraphrase-multilingual-MiniLM-L12-v2`
   - Index FAISS pour recherche rapide par similarité cosinus
   - Retriever sémantique avec scores de pertinence

3. **Intégration LLM**
   - Support de Qwen2.5 (0.5B, 1.5B, base)
   - Prompt engineering avec instructions claires
   - Génération contextualisée avec citations

4. **Détection hors domaine**
   - Seuil de similarité configurable
   - Réponses adaptées pour questions hors sujet

5. **Évaluation**
   - Précision, Recall, F1 pour le retrieval
   - ROUGE-L pour la qualité de génération

6. **Interface utilisateur**
   - Interface Gradio interactive
   - Sélection du modèle
   - Affichage des sources

### 📊 Performances attendues

- **Retrieval Precision**: ~0.6-0.8 selon la qualité des données
- **Retrieval Recall**: ~0.5-0.7
- **ROUGE-L**: ~0.3-0.5 selon la similarité avec les références

### 🔧 Améliorations possibles

1. Fine-tuning du LLM sur le domaine juridique marocain
2. Utilisation de modèles plus performants (Mistral, Llama 3)
3. Ajout de metadata supplémentaires (dates, catégories)
4. Support de questions multi-tours (conversation)
5. Cache des embeddings pour accélération

---